[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/34_representation_learning/90_exercises.ipynb)

# Bài tập — Chương 34: Representation Learning (L0 → L7)

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/34_representation_learning

In [ ]:
import math
import time

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import KNeighborsClassifier

from ml_course.checks import check_close, check_equal, check_greater, check_less, check_true
from ml_course.datasets import SHAPE_CLASSES_EXTRA, make_shapes

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
except ImportError as e:
    raise ImportError("Bài tập này cần PyTorch: pip install -e \".[dl]\"") from e

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
to_tensor = lambda X: torch.tensor(X[:, None], dtype=torch.float32)
X_unlabeled, y_hidden = make_shapes(20000, seed=0)
X_test, y_test = make_shapes(2000, seed=1)
X_unlabeled_t, X_test_t = to_tensor(X_unlabeled), to_tensor(X_test)


def nt_xent(z1, z2, tau=0.2):
    z = F.normalize(torch.cat([z1, z2]), dim=1)
    sim = z @ z.T / tau
    sim.fill_diagonal_(float("-inf"))
    n = len(z1)
    return F.cross_entropy(sim, torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)]).to(z.device))


class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        def block(i, o):
            return [nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU()]

        self.net = nn.Sequential(*block(1, 32), nn.MaxPool2d(2), *block(32, 64), nn.MaxPool2d(2), *block(64, 128),
                                 nn.AdaptiveAvgPool2d(1), nn.Flatten())

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def features(encoder, X):
    encoder.eval()
    return torch.cat([encoder(X[i:i + 1000].to(device)).cpu() for i in range(0, len(X), 1000)]).numpy()


def linear_probe(F_pool, F_test, n, seeds=range(3)):
    scores = []
    for s in seeds:
        idx = np.random.default_rng(s).choice(len(F_pool), n, replace=False)
        scores.append((LogisticRegression(max_iter=5000).fit(F_pool[idx], y_hidden[idx]).predict(F_test) == y_test).mean())
    return float(np.mean(scores))

## L0 — Nhớ khái niệm

**Câu 1.** (a) Autoencoder học gì; quan hệ với PCA? (b) Viết ELBO của VAE và vai trò mỗi số hạng. (c) Reparameterization trick? (d) Mô tả loss NT-Xent của SimCLR. (e) Linear probe là gì và vì sao dùng để đánh giá biểu diễn? (f) "Sụp đổ biểu diễn" là gì?

✍️ Trả lời:

## L1 — Giải thích bằng lời

**Câu 2.** Một sàn thương mại điện tử train autoencoder trên 2 triệu ảnh sản phẩm và dùng latent để tìm "sản phẩm tương tự". Kết quả trả về thường là sản phẩm **khác loại** nhưng cùng phông nền trắng, cùng vị trí đặt và cùng màu chủ đạo. Giải thích và đề xuất.

✍️ Trả lời:

## L2 — Tính tay

**Câu 3.** Encoder của VAE cho một chiều latent $\mu = 1$, $\sigma = 0.5$. Tính `kl` = KL$(\mathcal N(\mu, \sigma^2)\,\|\,\mathcal N(0, 1)) = \frac12(\sigma^2 + \mu^2 - 1 - \ln\sigma^2)$, và mẫu `z` theo reparameterization với $\varepsilon = -0.4$.

In [ ]:
### START CODE HERE ### (≈ 2 dòng)
kl = None
z = None
### END CODE HERE ###
print(f"KL = {kl:.4f} | z = {z}")

In [ ]:
ref3 = torch.distributions.kl_divergence(torch.distributions.Normal(1.0, 0.5), torch.distributions.Normal(0.0, 1.0)).item()
check_close(kl, ref3, message="khớp torch.distributions.kl_divergence")
check_close(z, 0.8, message="z = μ + σ·ε")

**Câu 4.** Batch 2 ảnh, nhiệt độ $\tau = 1$; vector chiếu (đã chuẩn hoá): ảnh 1 có hai góc nhìn $[1, 0]$ và $[1, 0]$, ảnh 2 có hai góc nhìn $[0, 1]$ và $[0, 1]$. Tính `loss` NT-Xent (giống nhau cho cả 4 góc nhìn): $-\log\frac{e^{1}}{e^{1} + e^{0} + e^{0}}$.

In [ ]:
### START CODE HERE ### (≈ 1 dòng)
loss = None
### END CODE HERE ###
print(f"NT-Xent = {loss:.4f}")

In [ ]:
z1, z2 = torch.tensor([[1.0, 0.0], [0.0, 1.0]]), torch.tensor([[1.0, 0.0], [0.0, 1.0]])
check_close(loss, nt_xent(z1, z2, tau=1.0).item(), message="khớp nt_xent")
check_less(nt_xent(z1, z2, tau=0.1).item(), loss, "τ nhỏ hơn → loss nhỏ hơn khi cặp dương đã khớp hoàn hảo")

## L3 — Code

**Bài 5.** Viết `reparameterize(mu, logvar, eps)` và `kl_to_standard_normal(mu, logvar)` (tổng theo chiều latent, trả về tensor shape (B,)). Kiểm tra: khớp `torch.distributions`, và **ước lượng Monte Carlo** của KL (trung bình $\log q(z) - \log p(z)$ trên nhiều mẫu $z$) phải xấp xỉ công thức đóng.

In [ ]:
def reparameterize(mu, logvar, eps):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###


def kl_to_standard_normal(mu, logvar):
    ### START CODE HERE ### (≈ 1 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
torch.manual_seed(0)
mu5, logvar5 = torch.randn(4, 3), torch.randn(4, 3)
q5 = torch.distributions.Normal(mu5, torch.exp(0.5 * logvar5))
p5 = torch.distributions.Normal(torch.zeros(4, 3), torch.ones(4, 3))
check_close(kl_to_standard_normal(mu5, logvar5), torch.distributions.kl_divergence(q5, p5).sum(1), tol=1e-5, message="KL công thức đóng")
eps5 = torch.randn(200000, 4, 3)
z5 = reparameterize(mu5, logvar5, eps5)
mc = (q5.log_prob(z5) - p5.log_prob(z5)).sum(-1).mean(0)
check_close(mc, kl_to_standard_normal(mu5, logvar5), tol=0.05, message="ước lượng Monte Carlo ≈ công thức đóng")
check_close(z5.mean(0), mu5, tol=0.02, message="E[z] = μ")

**Bài 6.** Đánh giá biểu diễn bằng **kNN** (phổ biến trong self-supervised, không cần train gì): viết `knn_predict(F_train, y_train, F_query, k)` — độ tương đồng cosine, lấy $k$ láng giềng gần nhất, bỏ phiếu đa số (hoà thì chọn nhãn nhỏ nhất). So khớp `KNeighborsClassifier(metric="cosine")`.

In [ ]:
def knn_predict(F_train, y_train, F_query, k=5):
    ### START CODE HERE ### (≈ 5 dòng)
    return None
    ### END CODE HERE ###

In [ ]:
r6 = np.random.default_rng(0)
F6, y6, Q6 = r6.normal(size=(300, 16)), r6.integers(0, 4, 300), r6.normal(size=(50, 16))
ref6 = KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute").fit(F6, y6).predict(Q6)
check_true(np.array_equal(knn_predict(F6, y6, Q6, 5), ref6), "khớp KNeighborsClassifier(metric='cosine')")
pix = X_unlabeled[:5000].reshape(5000, -1)
acc6 = (knn_predict(pix, y_hidden[:5000], X_test.reshape(2000, -1), 5) == y_test).mean()
print(f"kNN trên pixel thô (5 000 ảnh có nhãn): accuracy = {acc6:.3f}")
check_less(acc6, 0.7, "pixel thô: kNN kém — khoảng cách pixel không phản ánh loại hình")

## L4 — Debug

**Bài 7.** Một VAE (cho sẵn) cho mọi ảnh tái tạo **gần như giống hệt nhau** — một đốm mờ trung bình — và latent không mang thông tin gì. Lỗi nằm ở hàm loss. Tìm và sửa `vae_loss`; so sánh số chiều latent "hoạt động" (độ lệch chuẩn của $\mu$ trên tập test > 0.1) và MSE tái tạo.

In [ ]:
class VAE(nn.Module):
    def __init__(self, k=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(), nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(), nn.Flatten(), nn.Linear(64 * 7 * 7, 2 * k))
        self.decoder = nn.Sequential(nn.Linear(k, 64 * 7 * 7), nn.ReLU(), nn.Unflatten(1, (64, 7, 7)),
                                     nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(), nn.ConvTranspose2d(32, 1, 4, 2, 1))

    def forward(self, x):
        mu, logvar = self.encoder(x).chunk(2, dim=1)
        return self.decoder(mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)), mu, logvar


def vae_loss_buggy(logits, x, mu, logvar):
    recon = F.binary_cross_entropy_with_logits(logits, x)                       # trung bình theo PIXEL
    kl = (-0.5 * (1 + logvar - mu**2 - logvar.exp()).sum(1)).mean()            # tổng theo chiều latent
    return recon + kl


def vae_loss(logits, x, mu, logvar):
    ### START CODE HERE ### (≈ 3 dòng)
    return vae_loss_buggy(logits, x, mu, logvar)
    ### END CODE HERE ###


def train_vae(loss_fn, steps=2000, seed=0):
    torch.manual_seed(seed)
    model = VAE().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        xb = X_unlabeled_t[torch.randint(0, len(X_unlabeled_t), (128,), generator=g)].to(device)
        logits, mu, logvar = model(xb)
        opt.zero_grad()
        loss_fn(logits, xb, mu, logvar).backward()
        opt.step()
    return model.eval()


results7 = {}
for name, fn in [("lỗi", vae_loss_buggy), ("đã sửa", vae_loss)]:
    m7 = train_vae(fn)
    with torch.no_grad():
        _, mu7, _ = m7(X_test_t.to(device))
        mse7 = F.mse_loss(torch.sigmoid(m7.decoder(mu7)), X_test_t.to(device)).item()
    results7[name] = (int((mu7.std(0) > 0.1).sum()), mse7)
    print(f"loss {name:6s}: số chiều latent hoạt động = {results7[name][0]:2d}/16 | MSE tái tạo = {mse7:.4f}")

In [ ]:
check_equal(results7["lỗi"][0], 0, "tái hiện: posterior collapse — không chiều nào được dùng")
check_greater(results7["đã sửa"][0], 8, "đã sửa: latent được dùng")
check_less(results7["đã sửa"][1], results7["lỗi"][1] / 2, "đã sửa: tái tạo tốt hơn hẳn")

## L5 — Phân tích

**Bài 8.** **Augmentation nào tạo nên biểu diễn tốt?** Train SimCLR (1 000 bước, batch 256, τ = 0.2) với 4 cấu hình augmentation hình học: không có / chỉ dịch + lật / chỉ xoay / xoay + dịch + lật (luôn kèm độ sáng + nhiễu). Lưu `probe8[(xoay, dịch)]` = accuracy linear probe với 50 nhãn. Nhận xét.

In [ ]:
def augment(x, g, rotate, shift):
    B = x.shape[0]
    angle = (torch.rand(B, generator=g) * 2 - 1) * (math.pi if rotate else 0.0)
    offset = (torch.rand(2, B, generator=g) * 2 - 1) * (0.3 if shift else 0.0)
    flip = torch.where(torch.rand(B, generator=g) < 0.5, -1.0, 1.0) if (shift or rotate) else torch.ones(B)
    cos, sin = angle.cos(), angle.sin()
    theta = torch.stack([torch.stack([cos * flip, -sin, offset[0]], 1), torch.stack([sin * flip, cos, offset[1]], 1)], 1)
    x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), align_corners=False)
    return (x * (0.6 + 0.8 * torch.rand(B, 1, 1, 1, generator=g)) + 0.15 * torch.randn(x.shape, generator=g)).clamp(0, 1)


def train_simclr(rotate, shift, steps=1000, seed=0):
    torch.manual_seed(seed)
    encoder = Encoder().to(device)
    head = nn.Sequential(nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, 64)).to(device)
    opt = torch.optim.Adam(list(encoder.parameters()) + list(head.parameters()), lr=1e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        encoder.train()
        xb = X_unlabeled_t[torch.randint(0, len(X_unlabeled_t), (256,), generator=g)]
        opt.zero_grad()
        nt_xent(head(encoder(augment(xb, g, rotate, shift).to(device))), head(encoder(augment(xb, g, rotate, shift).to(device)))).backward()
        opt.step()
    return encoder.eval()


probe8, encoders8 = {}, {}
### START CODE HERE ### (≈ 6 dòng)
# với (rotate, shift) trong 4 cấu hình: train_simclr → features 5 000 ảnh đầu & tập test → linear_probe(..., 50)
pass
### END CODE HERE ###

In [ ]:
check_greater(probe8[(True, True)], probe8[(False, False)] + 0.1, "augmentation hình học cải thiện rõ")
check_greater(probe8[(True, True)], max(probe8[(True, False)], probe8[(False, True)]) - 0.01, "kết hợp tốt nhất (hoặc ngang)")
check_greater(min(probe8[(True, False)], probe8[(False, True)]), probe8[(False, False)], "mỗi loại đều giúp")

## L6 — Thực tế

**Bài 9.** **Tìm ảnh tương tự** bằng embedding (dùng encoder SimCLR tốt nhất của bài 8). Với mỗi ảnh truy vấn, lấy 10 ảnh gần nhất (cosine) trong cùng tập (trừ chính nó); **precision@10** = tỉ lệ ảnh trả về cùng loại hình. Tính `p10[(biểu diễn, tập)]` cho biểu diễn ∈ {"pixel", "SimCLR"} và tập ∈ {"hình cũ" (2 000 ảnh test), "hình mới" (2 000 ảnh của 4 hình **chưa từng thấy** — `SHAPE_CLASSES_EXTRA`)}.

In [ ]:
X_new, y_new = make_shapes(2000, seed=30, classes=SHAPE_CLASSES_EXTRA)
X_new_t = to_tensor(X_new)


def precision_at_10(F_all, y):
    ### START CODE HERE ### (≈ 4 dòng)
    return None
    ### END CODE HERE ###


best_encoder = encoders8[(True, True)]
p10 = {}
for name, (X_np, X_t, y) in {"hình cũ": (X_test, X_test_t, y_test), "hình mới": (X_new, X_new_t, y_new)}.items():
    p10[("pixel", name)] = precision_at_10(X_np.reshape(len(X_np), -1), y)
    p10[("SimCLR", name)] = precision_at_10(features(best_encoder, X_t), y)
for key, v in p10.items():
    print(f"{key[0]:7s} | {key[1]:8s}: precision@10 = {v:.3f}")

In [ ]:
check_close(precision_at_10(np.eye(3)[[0, 0, 1, 1, 2, 2] * 4], np.array([0, 0, 1, 1, 2, 2] * 4)), 0.7, message="ví dụ nhỏ: 7 ảnh cùng lớp (trừ chính nó) trong 10 láng giềng")
check_greater(p10[("SimCLR", "hình cũ")], p10[("pixel", "hình cũ")] + 0.3, "SimCLR vượt pixel trên hình cũ")
check_greater(p10[("SimCLR", "hình mới")], 0.8, "embedding tổng quát hoá sang hình CHƯA TỪNG THẤY")

## L7 — Nâng cao

**Bài 10.** **Phát hiện bất thường: biểu diễn quan trọng thế nào?** Tập kiểm tra: 1 000 ảnh 5 hình đã biết + 1 000 ảnh 4 hình mới (bất thường). So sánh hai điểm bất thường: (a) **sai số tái tạo** của một autoencoder tích chập (latent 32, train 2 000 bước trên dữ liệu không nhãn); (b) **khoảng cách kNN** trong không gian SimCLR: $1 -$ trung bình cosine với 5 láng giềng gần nhất trong 5 000 ảnh "bình thường". Lưu `auc10["autoencoder"]`, `auc10["SimCLR kNN"]`.

In [ ]:
X_new10, _ = make_shapes(1000, seed=3, classes=SHAPE_CLASSES_EXTRA)
X_mix = torch.cat([X_test_t[:1000], to_tensor(X_new10)])
is_anomaly = np.r_[np.zeros(1000), np.ones(1000)]
auc10 = {}
### START CODE HERE ### (≈ 18 dòng)
# (a) train ConvAE → sai số tái tạo từng ảnh của X_mix; (b) features SimCLR → 1 − trung bình cosine top-5 với 5 000 ảnh bình thường
pass
### END CODE HERE ###
for k, v in auc10.items():
    print(f"{k:12s}: ROC-AUC = {v:.3f}")

In [ ]:
check_greater(auc10["SimCLR kNN"], auc10["autoencoder"] + 0.05, "biểu diễn ngữ nghĩa phát hiện hình mới tốt hơn sai số tái tạo")
check_greater(auc10["SimCLR kNN"], 0.75, "kNN trên SimCLR")